In [1]:
# -*- coding: utf-8 -*-
import os
import re
import glob
import json
import warnings
import numpy as np
import pandas as pd

import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score

# 可选：保存最终模型
import joblib

# =========================
# 0) 全局：尽量屏蔽警告
# =========================
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# =========================
# 1) 配置
# =========================
RANDOM_SEED = 42
N_SPLITS = 5
THRESH = 0.5

FEATURE_FILE_GLOB = "./Malodors_transformed_MORGAN_features.xlsx"

# 你 Optuna 搜索输出的结果文件（用于读取 best params）
OPTUNA_RESULTS_CSV = "./Morgan_XGB_5fold.csv"

# 输出文件
OUT_FOLDS_CSV = "./xgb_bestparams_5fold_metrics.csv"
OUT_MEANCI_CSV = "./xgb_bestparams_5fold_mean_ci95.csv"
OUT_MODEL_FILE = "./xgb_bestparams_fullfit.joblib"

# 固定参数（会与 best_params 合并）
BASE_XGB_PARAMS = dict(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbosity=0,
    multi_strategy="multi_output_tree",  # 需要 xgboost>=2.0
)

# =========================
# 2) 版本检查
# =========================
def _ver_tuple(v: str):
    parts = re.split(r"[.+-]", v.strip())
    nums = []
    for p in parts[:3]:
        try:
            nums.append(int(p))
        except Exception:
            nums.append(0)
    while len(nums) < 3:
        nums.append(0)
    return tuple(nums)

def check_xgb_version():
    v = _ver_tuple(xgb.__version__)
    if v < (1, 6, 0):
        raise RuntimeError(f"xgboost=={xgb.__version__} 太旧：multi-label 需要 >=1.6")
    if "multi_strategy" in BASE_XGB_PARAMS and v < (2, 0, 0):
        raise RuntimeError(
            f"xgboost=={xgb.__version__} 不支持 multi_output_tree（需要>=2.0）。"
            f"升级或删掉 multi_strategy。"
        )

# =========================
# 3) 读取 transformed 特征文件 & 自动识别列
# =========================
def find_feature_file(pattern: str):
    cands = sorted(glob.glob(pattern))
    if not cands:
        raise FileNotFoundError(
            f"找不到特征文件：{pattern}\n"
            f"请确认你保存的 xlsx 文件名，或修改 FEATURE_FILE_GLOB。"
        )
    return cands[-1]

def find_smiles_col(df: pd.DataFrame):
    cand = [c for c in df.columns if isinstance(c, str) and "smiles" in c.lower()]
    if not cand:
        return None
    for p in ["Canonical SMILES", "canonical_smiles", "SMILES", "smiles"]:
        for c in cand:
            if c.lower() == p.lower():
                return c
    return cand[0]

def is_binary_01_series(s: pd.Series) -> bool:
    if s.dtype == bool:
        return True
    if not np.issubdtype(s.dtype, np.number):
        return False
    vals = pd.unique(s.dropna())
    if len(vals) == 0:
        return False
    return set(vals).issubset({0, 1})

def infer_feature_cols_morgan(df: pd.DataFrame):
    """
    兼容：
    - 前缀列：Rule__/FG__/morgan_/fp_/bit_/KG_emb_ ...
    - 纯数字列：0..2047（Excel 读入后可能变 int 或 str）
    """
    cols = []
    for prefix in ["Rule__", "FG__", "morgan_", "fp_", "ecfp_", "mfp_", "bit_", "KG_emb_"]:
        cols.extend([c for c in df.columns if isinstance(c, str) and c.startswith(prefix)])
    if cols:
        return list(cols)

    num_cols = []
    for c in df.columns:
        if isinstance(c, (int, np.integer)):
            num_cols.append(c)
        elif isinstance(c, str) and c.isdigit():
            num_cols.append(c)
    if num_cols:
        return sorted(num_cols, key=lambda x: int(x))

    raise ValueError("未找到特征列（Rule__/FG__/morgan_/bit_/KG_emb_ 或 0..2047 纯数字列）。")

def infer_label_cols(df: pd.DataFrame, smiles_col: str, feature_cols: list):
    exclude = set(feature_cols)
    if smiles_col is not None:
        exclude.add(smiles_col)

    label_cols = []
    for c in df.columns:
        if c in exclude:
            continue
        if is_binary_01_series(df[c]):
            label_cols.append(c)

    if not label_cols:
        raise ValueError("未识别到标签列（0/1）。请确认文件中仍包含 138 个标签列。")
    return label_cols

def load_Xy_from_transformed_morgan(xlsx_path: str):
    df = pd.read_excel(xlsx_path)

    smiles_col = find_smiles_col(df)
    feature_cols = infer_feature_cols_morgan(df)
    label_cols = infer_label_cols(df, smiles_col, feature_cols)

    X = df[feature_cols].fillna(0).astype(np.float32).values
    y = df[label_cols].fillna(0).astype(int).values

    if X.shape[0] != y.shape[0]:
        raise ValueError("X 和 y 行数不一致，请检查文件。")

    return X, y, feature_cols, label_cols, df

# =========================
# 4) 指标（macro）
# =========================
def multilabel_macro_metrics(y_true, y_prob, thresh=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= thresh).astype(int)

    L = y_true.shape[1]
    accs, precs, recs, specs = [], [], [], []
    aurocs, auprcs = [], []

    for k in range(L):
        yt = y_true[:, k]
        yp = y_pred[:, k]
        ys = y_prob[:, k]

        tp = int(np.sum((yt == 1) & (yp == 1)))
        tn = int(np.sum((yt == 0) & (yp == 0)))
        fp = int(np.sum((yt == 0) & (yp == 1)))
        fn = int(np.sum((yt == 1) & (yp == 0)))
        n = len(yt)

        accs.append((tp + tn) / n if n else np.nan)
        precs.append(tp / (tp + fp) if (tp + fp) else np.nan)
        recs.append(tp / (tp + fn) if (tp + fn) else np.nan)
        specs.append(tn / (tn + fp) if (tn + fp) else np.nan)

        if len(np.unique(yt)) == 2:
            aurocs.append(roc_auc_score(yt, ys))
            auprcs.append(average_precision_score(yt, ys))
        else:
            aurocs.append(np.nan)
            auprcs.append(np.nan)

    def nanmean(x):
        return float(np.nanmean(np.asarray(x, dtype=float)))

    return {
        "Accuracy_macro": nanmean(accs),
        "Precision_macro": nanmean(precs),
        "Recall_macro": nanmean(recs),
        "Specificity_macro": nanmean(specs),
        "AUROC_macro": nanmean(aurocs),
        "AUPRC_macro": nanmean(auprcs),
        "AUROC_valid_labels": int(np.sum(~np.isnan(aurocs))),
        "AUPRC_valid_labels": int(np.sum(~np.isnan(auprcs))),
        "n_labels": int(L),
    }

def extract_positive_proba(p, n_labels: int):
    """
    multi-output XGBClassifier predict_proba 可能是：
    - ndarray (n, L)         : 已是正类概率
    - ndarray (n, L, 2)      : 二分类概率
    - list length=L, (n,2)   : 每标签一组概率
    """
    if isinstance(p, list):
        return np.vstack([pi[:, 1] for pi in p]).T.astype(np.float32)

    p = np.asarray(p)
    if p.ndim == 3 and p.shape[-1] == 2:
        return p[:, :, 1].astype(np.float32)
    if p.ndim == 2:
        if p.shape[1] == 2 and n_labels == 1:
            return p[:, 1:2].astype(np.float32)
        return p.astype(np.float32)
    raise ValueError(f"无法解析 predict_proba 输出形状: {p.shape}")

# =========================
# 5) 近似分层 5 折：用 label cardinality 分层
# =========================
def make_stratify_target(y: np.ndarray, n_bins: int = 10):
    card = y.sum(axis=1).astype(int)
    uniq = np.unique(card)
    if len(uniq) <= 15:
        return card
    r = pd.Series(card).rank(method="average").values
    bins = pd.qcut(r, q=min(n_bins, len(np.unique(r))), labels=False, duplicates="drop")
    return np.asarray(bins, dtype=int)

def build_folds(X, y, n_splits=5, seed=42):
    strat_y = make_stratify_target(y)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return list(skf.split(X, strat_y))

# =========================
# 6) 从 Optuna 结果 CSV 读取最优超参 + 类型修复
# =========================
def load_best_params_from_optuna_csv(csv_path: str):
    df = pd.read_csv(csv_path)
    if "AUPRC_macro" not in df.columns:
        raise ValueError(f"Optuna 结果文件缺少 AUPRC_macro 列：{csv_path}")

    best_row = df.sort_values("AUPRC_macro", ascending=False).iloc[0].to_dict()

    drop_cols = set([
        "trial", "AUPRC_macro",
        "AUROC_macro", "Accuracy_macro", "Precision_macro", "Recall_macro", "Specificity_macro",
    ])

    params = {}
    for k, v in best_row.items():
        if k in drop_cols:
            continue
        params[k] = v  # 先原样取出，后面统一 sanitize

    # 合并固定参数
    final_params = dict(BASE_XGB_PARAMS)
    final_params.update(params)

    # --- 关键：类型修复（防止 n_estimators/max_depth 变 float） ---
    final_params = sanitize_xgb_params(final_params)

    return final_params

def sanitize_xgb_params(params: dict) -> dict:
    """
    修复从 CSV 读出来的 best_params 类型问题：
    - n_estimators / max_depth / n_jobs / random_state 必须 int
    - 其它本应 float 的保持 float
    - str/bool 保持
    """
    p = dict(params)

    # 必须是 int 的参数
    int_keys = ["n_estimators", "max_depth", "n_jobs", "random_state"]
    for k in int_keys:
        if k in p and p[k] is not None and not (isinstance(p[k], str) and p[k].strip() == ""):
            try:
                p[k] = int(float(p[k]))
            except Exception:
                pass

    # 必须是 float 的参数
    float_keys = [
        "learning_rate", "subsample", "colsample_bytree",
        "min_child_weight", "reg_lambda", "reg_alpha", "gamma"
    ]
    for k in float_keys:
        if k in p and p[k] is not None and not (isinstance(p[k], str) and p[k].strip() == ""):
            try:
                p[k] = float(p[k])
            except Exception:
                pass

    return p

# =========================
# 7) 五折CV评估（用固定最优超参）
# =========================
def cv_eval_bestparams(X, y, folds, params, thresh=0.5):
    n_labels = y.shape[1]
    fold_rows = []

    for fold_id, (tr_idx, va_idx) in enumerate(folds, start=1):
        X_tr, X_va = X[tr_idx], X[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        clf = xgb.XGBClassifier(**params)
        clf.fit(X_tr, y_tr)

        p = clf.predict_proba(X_va)
        y_prob = extract_positive_proba(p, n_labels=n_labels)

        m = multilabel_macro_metrics(y_va, y_prob, thresh=thresh)
        m["fold"] = fold_id
        fold_rows.append(m)

        print(
            f"[FOLD {fold_id}] "
            f"AUPRC={m['AUPRC_macro']:.6f} | AUROC={m['AUROC_macro']:.6f} | "
            f"Acc={m['Accuracy_macro']:.6f} | P={m['Precision_macro']:.6f} | "
            f"R={m['Recall_macro']:.6f} | Spec={m['Specificity_macro']:.6f}"
        )

    return pd.DataFrame(fold_rows).sort_values("fold")

# =========================
# 8) mean+_CI95（小样本用t；无scipy则退回1.96）
# =========================
def mean_ci95(arr: np.ndarray):
    arr = np.asarray(arr, dtype=float)
    arr = arr[~np.isnan(arr)]
    n = len(arr)
    if n == 0:
        return np.nan, np.nan, np.nan, 0

    mean = float(np.mean(arr))
    std = float(np.std(arr, ddof=1)) if n >= 2 else 0.0
    se = std / np.sqrt(n) if n > 0 else np.nan

    try:
        import scipy.stats as st
        tcrit = float(st.t.ppf(0.975, df=n-1)) if n >= 2 else 1.96
    except Exception:
        tcrit = 1.96

    half = tcrit * se if n >= 2 else 0.0
    return mean, mean - half, mean + half, n

def format_mean_plus_ci95(mean, lo, hi):
    if np.isnan(mean):
        return "nan"
    half = (hi - lo) / 2.0
    return f"{mean:.6f}+_{half:.6f}"

def summarize_mean_ci95(folds_df: pd.DataFrame):
    metric_cols = [
        "Accuracy_macro",
        "Precision_macro",
        "Recall_macro",
        "Specificity_macro",
        "AUROC_macro",
        "AUPRC_macro",
    ]
    rows = []
    for col in metric_cols:
        mean, lo, hi, n = mean_ci95(folds_df[col].values)
        rows.append({
            "metric": col,
            "mean": mean,
            "ci95_low": lo,
            "ci95_high": hi,
            "n_folds": n,
            "mean+_CI95": format_mean_plus_ci95(mean, lo, hi),
        })
    return pd.DataFrame(rows)

# =========================
# 9) 主流程：读特征 -> 读最优参数 -> 5折CV -> 输出&保存 -> 全量fit保存
# =========================
def main():
    check_xgb_version()

    feature_file = find_feature_file(FEATURE_FILE_GLOB)
    print("[INFO] Using transformed Morgan feature file:", feature_file)

    X, y, feat_cols, label_cols, _df_all = load_Xy_from_transformed_morgan(feature_file)
    print(f"[INFO] X shape={X.shape} (features={len(feat_cols)}) | y shape={y.shape} (labels={len(label_cols)})")

    best_params = load_best_params_from_optuna_csv(OPTUNA_RESULTS_CSV)
    print("\n[INFO] Loaded best params from Optuna CSV (sanitized):")
    print(json.dumps(best_params, indent=2, ensure_ascii=False, default=str))

    folds = build_folds(X, y, n_splits=N_SPLITS, seed=RANDOM_SEED)
    print(f"\n[INFO] Prepared fixed {N_SPLITS}-fold splits.")

    # 5-fold CV
    folds_df = cv_eval_bestparams(X, y, folds, best_params, thresh=THRESH)
    folds_df.to_csv(OUT_FOLDS_CSV, index=False, encoding="utf-8-sig")
    print(f"\n[SAVED] {OUT_FOLDS_CSV}")

    # mean+_CI95 汇总
    sum_df = summarize_mean_ci95(folds_df)
    sum_df.to_csv(OUT_MEANCI_CSV, index=False, encoding="utf-8-sig")
    print(f"[SAVED] {OUT_MEANCI_CSV}")

    print("\n========== 5-FOLD MEAN+_CI95 ==========")
    for _, r in sum_df.iterrows():
        print(f"{r['metric']}: {r['mean+_CI95']}")

    # 全量训练并保存
    clf_full = xgb.XGBClassifier(**best_params)
    clf_full.fit(X, y)

    joblib.dump(
        {
            "model": clf_full,
            "params": best_params,
            "feature_cols": feat_cols,
            "label_cols": label_cols,
            "threshold": THRESH,
            "random_seed": RANDOM_SEED,
        },
        OUT_MODEL_FILE
    )
    print(f"\n[SAVED] full-fit model -> {OUT_MODEL_FILE}")

if __name__ == "__main__":
    main()


[INFO] Using transformed Morgan feature file: ./Malodors_transformed_MORGAN_features.xlsx
[INFO] X shape=(4952, 2048) (features=2048) | y shape=(4952, 138) (labels=138)

[INFO] Loaded best params from Optuna CSV (sanitized):
{
  "objective": "binary:logistic",
  "eval_metric": "logloss",
  "tree_method": "hist",
  "n_jobs": -1,
  "random_state": 42,
  "verbosity": 0,
  "multi_strategy": "multi_output_tree",
  "n_estimators": 1099,
  "max_depth": 8,
  "learning_rate": 0.018522477663361,
  "subsample": 0.8686080349896406,
  "colsample_bytree": 0.9520104938890754,
  "min_child_weight": 1.8942028945930036,
  "reg_lambda": 0.0167040981359912,
  "reg_alpha": 0.0019461130896047,
  "gamma": 1.84787157882324
}

[INFO] Prepared fixed 5-fold splits.
[FOLD 1] AUPRC=0.307396 | AUROC=0.865863 | Acc=0.973128 | P=0.560842 | R=0.150099 | Spec=0.994137
[FOLD 2] AUPRC=0.327762 | AUROC=0.871623 | Acc=0.972843 | P=0.580831 | R=0.149561 | Spec=0.993772
[FOLD 3] AUPRC=0.306965 | AUROC=0.863215 | Acc=0.972574

In [2]:
# -*- coding: utf-8 -*-
import os
import re
import glob
import json
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier

import joblib

# =========================
# 0) 全局：尽量屏蔽警告
# =========================
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# =========================
# 1) 配置
# =========================
RANDOM_SEED = 42
N_SPLITS = 5
THRESH = 0.5

# 你必须有这个：特征文件匹配
FEATURE_FILE_GLOB = "./Malodors_transformed_MORGAN_features.xlsx"  # 改成你的

OUT_FOLDS_CSV   = "./rf_bestparams_5fold_metrics.csv"
OUT_MEANCI_CSV  = "./rf_bestparams_5fold_mean_ci95.csv"
OUT_MODEL_FILE  = "./rf_bestparams_fullfit.joblib"

# ✅ 直接指定 best params（不再读 CSV）
BEST_PARAMS = {
  "max_depth": 35,
  "n_estimators": 1280,
  "max_features": "sqrt",
  "min_samples_split": 5,
  "min_samples_leaf": 3,
  "bootstrap": False,
  "criterion": "gini"
}

# RandomForest 固定参数（补齐 n_jobs / random_state）
BASE_RF_PARAMS = dict(
    n_jobs=-1,
    random_state=RANDOM_SEED,
)

# =========================
# 2) 读特征文件
# =========================
def find_feature_file(pattern: str):
    cands = sorted(glob.glob(pattern))
    if not cands:
        raise FileNotFoundError(
            f"找不到特征文件：{pattern}\n"
            f"请确认 xlsx 文件名，或修改 FEATURE_FILE_GLOB。"
        )
    return cands[-1]

def find_smiles_col(df: pd.DataFrame):
    cand = [c for c in df.columns if isinstance(c, str) and "smiles" in c.lower()]
    if not cand:
        return df.columns[0]
    for p in ["Canonical SMILES", "canonical_smiles", "SMILES", "smiles"]:
        for c in cand:
            if c.lower() == p.lower():
                return c
    return cand[0]

def is_binary_01_series(s: pd.Series) -> bool:
    if s.dtype == bool:
        return True
    if not np.issubdtype(s.dtype, np.number):
        return False
    vals = pd.unique(s.dropna())
    if len(vals) == 0:
        return False
    return set(vals).issubset({0, 1})

def infer_feature_cols_morgan(df: pd.DataFrame):
    morgan_cols = []
    for prefix in ["Rule__", "FG__", "morgan_", "fp_", "ecfp_", "mfp_", "bit_", "KG_emb_"]:
        morgan_cols.extend([c for c in df.columns if isinstance(c, str) and c.startswith(prefix)])
    if morgan_cols:
        return list(morgan_cols)

    # 纯数字列
    num_cols = []
    for c in df.columns:
        if isinstance(c, (int, np.integer)):
            num_cols.append(c)
        elif isinstance(c, str) and c.isdigit():
            num_cols.append(c)
    if num_cols:
        return sorted(num_cols, key=lambda x: int(x))

    raise ValueError("未找到特征列（Rule__/FG__/morgan_/bit_/KG_emb_ 或 0..2047 纯数字列）。")

def infer_label_cols(df: pd.DataFrame, smiles_col: str, feature_cols: list):
    exclude = set(feature_cols)
    if smiles_col is not None:
        exclude.add(smiles_col)

    label_cols = []
    for c in df.columns:
        if c in exclude:
            continue
        if is_binary_01_series(df[c]):
            label_cols.append(c)

    if not label_cols:
        raise ValueError("未识别到标签列（0/1）。请确认文件中仍包含标签列。")
    return label_cols

def load_Xy_from_transformed_morgan(xlsx_path: str):
    df = pd.read_excel(xlsx_path)

    smiles_col = find_smiles_col(df)
    feature_cols = infer_feature_cols_morgan(df)
    label_cols = infer_label_cols(df, smiles_col, feature_cols)

    X = df[feature_cols].fillna(0).astype(np.float32).values
    y = df[label_cols].fillna(0).astype(int).values

    if X.shape[0] != y.shape[0]:
        raise ValueError("X 和 y 行数不一致，请检查文件。")

    return X, y, feature_cols, label_cols, df

# =========================
# 3) 指标（macro）
# =========================
def multilabel_macro_metrics(y_true, y_prob, thresh=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= thresh).astype(int)

    L = y_true.shape[1]
    accs, precs, recs, specs = [], [], [], []
    aurocs, auprcs = [], []

    for k in range(L):
        yt = y_true[:, k]
        yp = y_pred[:, k]
        ys = y_prob[:, k]

        tp = int(np.sum((yt == 1) & (yp == 1)))
        tn = int(np.sum((yt == 0) & (yp == 0)))
        fp = int(np.sum((yt == 0) & (yp == 1)))
        fn = int(np.sum((yt == 1) & (yp == 0)))
        n = len(yt)

        accs.append((tp + tn) / n if n else np.nan)
        precs.append(tp / (tp + fp) if (tp + fp) else np.nan)
        recs.append(tp / (tp + fn) if (tp + fn) else np.nan)
        specs.append(tn / (tn + fp) if (tn + fp) else np.nan)

        if len(np.unique(yt)) == 2:
            aurocs.append(roc_auc_score(yt, ys))
            auprcs.append(average_precision_score(yt, ys))
        else:
            aurocs.append(np.nan)
            auprcs.append(np.nan)

    def nanmean(x):
        return float(np.nanmean(np.asarray(x, dtype=float)))

    return {
        "Accuracy_macro": nanmean(accs),
        "Precision_macro": nanmean(precs),
        "Recall_macro": nanmean(recs),
        "Specificity_macro": nanmean(specs),
        "AUROC_macro": nanmean(aurocs),
        "AUPRC_macro": nanmean(auprcs),
        "AUROC_valid_labels": int(np.sum(~np.isnan(aurocs))),
        "AUPRC_valid_labels": int(np.sum(~np.isnan(auprcs))),
        "n_labels": int(L),
    }

def extract_positive_proba(p_list, n_labels: int, classes_list=None):
    """
    MultiOutputClassifier.predict_proba -> list length=L
    每个元素 pk: (n, 2) 或 (n,1)
    """
    if not isinstance(p_list, list):
        raise ValueError("MultiOutputClassifier.predict_proba 预期返回 list，但未得到 list。")

    out = np.zeros((p_list[0].shape[0], n_labels), dtype=np.float32)

    for k in range(n_labels):
        pk = p_list[k]
        if pk.ndim != 2:
            raise ValueError(f"predict_proba[{k}] 形状异常: {pk.shape}")

        if pk.shape[1] == 2:
            if classes_list is not None and len(classes_list) == n_labels:
                cls = list(classes_list[k])
                out[:, k] = pk[:, cls.index(1)].astype(np.float32) if 1 in cls else 0.0
            else:
                out[:, k] = pk[:, 1].astype(np.float32)

        elif pk.shape[1] == 1:
            if classes_list is not None and len(classes_list) == n_labels:
                only_cls = int(list(classes_list[k])[0])
                out[:, k] = 1.0 if only_cls == 1 else 0.0
            else:
                out[:, k] = 0.0
        else:
            raise ValueError(f"predict_proba[{k}] 类别数异常: {pk.shape[1]}")

    return out

# =========================
# 4) 近似分层 5 折：用 label cardinality 分层
# =========================
def make_stratify_target(y: np.ndarray, n_bins: int = 10):
    card = y.sum(axis=1).astype(int)
    uniq = np.unique(card)
    if len(uniq) <= 15:
        return card
    r = pd.Series(card).rank(method="average").values
    bins = pd.qcut(r, q=min(n_bins, len(np.unique(r))), labels=False, duplicates="drop")
    return np.asarray(bins, dtype=int)

def build_folds(X, y, n_splits=5, seed=42):
    strat_y = make_stratify_target(y)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return list(skf.split(X, strat_y))

# =========================
# 5) CV评估（固定 best params）
# =========================
def cv_eval_bestparams(X, y, folds, params, thresh=0.5):
    n_labels = y.shape[1]
    fold_rows = []

    for fold_id, (tr_idx, va_idx) in enumerate(folds, start=1):
        X_tr, X_va = X[tr_idx], X[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        base_est = RandomForestClassifier(**params)
        clf = MultiOutputClassifier(base_est, n_jobs=1)  # 外层不并行更稳
        clf.fit(X_tr, y_tr)

        p_list = clf.predict_proba(X_va)
        y_prob = extract_positive_proba(p_list, n_labels=n_labels, classes_list=getattr(clf, "classes_", None))

        m = multilabel_macro_metrics(y_va, y_prob, thresh=thresh)
        m["fold"] = fold_id
        fold_rows.append(m)

        print(
            f"[FOLD {fold_id}] "
            f"AUPRC={m['AUPRC_macro']:.6f} | AUROC={m['AUROC_macro']:.6f} | "
            f"Acc={m['Accuracy_macro']:.6f} | P={m['Precision_macro']:.6f} | "
            f"R={m['Recall_macro']:.6f} | Spec={m['Specificity_macro']:.6f}"
        )

    return pd.DataFrame(fold_rows).sort_values("fold")

# =========================
# 6) mean ± CI95
# =========================
def mean_ci95(arr: np.ndarray):
    arr = np.asarray(arr, dtype=float)
    arr = arr[~np.isnan(arr)]
    n = len(arr)
    if n == 0:
        return np.nan, np.nan, np.nan, 0
    mean = float(np.mean(arr))
    std = float(np.std(arr, ddof=1)) if n >= 2 else 0.0
    se = std / np.sqrt(n) if n > 0 else np.nan
    try:
        import scipy.stats as st
        tcrit = float(st.t.ppf(0.975, df=n-1)) if n >= 2 else 1.96
    except Exception:
        tcrit = 1.96
    half = tcrit * se if n >= 2 else 0.0
    return mean, mean - half, mean + half, n

def format_mean_ci95(mean, lo, hi):
    if np.isnan(mean):
        return "nan"
    half = (hi - lo) / 2.0
    return f"{mean:.6f} ± {half:.6f}"

def summarize_mean_ci95(folds_df: pd.DataFrame):
    metric_cols = [
        "Accuracy_macro",
        "Precision_macro",
        "Recall_macro",
        "Specificity_macro",
        "AUROC_macro",
        "AUPRC_macro",
    ]
    rows = []
    for col in metric_cols:
        mean, lo, hi, n = mean_ci95(folds_df[col].values)
        rows.append({
            "metric": col,
            "mean": mean,
            "ci95_low": lo,
            "ci95_high": hi,
            "n_folds": n,
            "mean±CI95": format_mean_ci95(mean, lo, hi),
        })
    return pd.DataFrame(rows)

# =========================
# 7) 主流程
# =========================
def main():
    feature_file = find_feature_file(FEATURE_FILE_GLOB)
    print("[INFO] Using feature file:", feature_file)

    X, y, feat_cols, label_cols, _df_all = load_Xy_from_transformed_morgan(feature_file)
    print(f"[INFO] X shape={X.shape} (features={len(feat_cols)}) | y shape={y.shape} (labels={len(label_cols)})")

    # ✅ 合并固定参数 + 你手写的 best params
    best_params = dict(BASE_RF_PARAMS)
    best_params.update(BEST_PARAMS)

    print("\n[INFO] Using fixed best params (directly specified):")
    print(json.dumps(best_params, indent=2, ensure_ascii=False))

    folds = build_folds(X, y, n_splits=N_SPLITS, seed=RANDOM_SEED)
    print(f"\n[INFO] Prepared fixed {N_SPLITS}-fold splits.")

    folds_df = cv_eval_bestparams(X, y, folds, best_params, thresh=THRESH)
    folds_df.to_csv(OUT_FOLDS_CSV, index=False, encoding="utf-8-sig")
    print(f"\n[SAVED] {OUT_FOLDS_CSV}")

    sum_df = summarize_mean_ci95(folds_df)
    sum_df.to_csv(OUT_MEANCI_CSV, index=False, encoding="utf-8-sig")
    print(f"[SAVED] {OUT_MEANCI_CSV}")

    print("\n========== 5-FOLD MEAN ± CI95 ==========")
    for _, r in sum_df.iterrows():
        print(f"{r['metric']}: {r['mean±CI95']}")

    # ✅ 全量 fit：保持和 CV 一致（MultiOutputClassifier）
    base_est = RandomForestClassifier(**best_params)
    clf_full = MultiOutputClassifier(base_est, n_jobs=1)
    clf_full.fit(X, y)

    joblib.dump(
        {
            "model": clf_full,
            "params": best_params,
            "feature_cols": feat_cols,
            "label_cols": label_cols,
            "threshold": THRESH,
            "random_seed": RANDOM_SEED,
        },
        OUT_MODEL_FILE
    )
    print(f"\n[SAVED] full-fit model -> {OUT_MODEL_FILE}")

if __name__ == "__main__":
    main()


[INFO] Using feature file: ./Malodors_transformed_MORGAN_features.xlsx
[INFO] X shape=(4952, 2048) (features=2048) | y shape=(4952, 138) (labels=138)

[INFO] Using fixed best params (directly specified):
{
  "n_jobs": -1,
  "random_state": 42,
  "max_depth": 35,
  "n_estimators": 1280,
  "max_features": "sqrt",
  "min_samples_split": 5,
  "min_samples_leaf": 3,
  "bootstrap": false,
  "criterion": "gini"
}

[INFO] Prepared fixed 5-fold splits.
[FOLD 1] AUPRC=0.283195 | AUROC=0.836043 | Acc=0.972389 | P=0.752815 | R=0.047501 | Spec=0.997752
[FOLD 2] AUPRC=0.308232 | AUROC=0.845367 | Acc=0.972148 | P=0.840400 | R=0.046312 | Spec=0.997758
[FOLD 3] AUPRC=0.290374 | AUROC=0.835549 | Acc=0.972127 | P=0.791390 | R=0.045962 | Spec=0.997601
[FOLD 4] AUPRC=0.306417 | AUROC=0.840005 | Acc=0.972178 | P=0.776111 | R=0.047513 | Spec=0.997769
[FOLD 5] AUPRC=0.292017 | AUROC=0.841350 | Acc=0.972347 | P=0.794321 | R=0.040819 | Spec=0.998144

[SAVED] ./rf_bestparams_5fold_metrics.csv
[SAVED] ./rf_bestpa

In [3]:
# -*- coding: utf-8 -*-
import os
import re
import glob
import json
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.multioutput import MultiOutputClassifier

import joblib

# =========================
# 0) 全局：尽量屏蔽警告 + LightGBM 日志
# =========================
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

try:
    import lightgbm as lgb
except Exception as e:
    raise RuntimeError(
        "未检测到 lightgbm。请先安装：pip install lightgbm\n"
        f"原始错误：{repr(e)}"
    )

# =========================
# 1) 配置
# =========================
RANDOM_SEED = 42
N_SPLITS = 5
THRESH = 0.5

  # 0/1阈值

# ✅ 你必须有：特征文件路径（按你自己的文件名改）
FEATURE_FILE_GLOB = "./Malodors_transformed_MORGAN_features.xlsx"

OUT_FOLDS_CSV  = "./lgb_bestparams_5fold_metrics.csv"
OUT_MEANCI_CSV = "./lgb_bestparams_5fold_mean_ci95.csv"
OUT_MODEL_FILE = "./lgb_bestparams_fullfit.joblib"

# 固定参数（会与 best_params 合并）
BASE_LGB_PARAMS = dict(
    objective="binary",
    boosting_type="gbdt",
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbosity=-1,
)

# ✅ 直接指定 best params（不再读 CSV）
BEST_PARAMS = {
  "n_estimators": 507,
  "learning_rate": 0.04407984038169244,
  "num_leaves": 38,
  "max_depth": 19,
  "min_child_samples": 24,
  "subsample": 0.8650089137415928,
  "colsample_bytree": 0.7246844304357644,
  "reg_lambda": 0.2778331300797557,
  "reg_alpha": 0.0008325158565947976,
  "min_split_gain": 0.9242722776276352
}

# =========================
# 2) 读取 transformed 特征文件 & 自动识别列
# =========================
def find_feature_file(pattern: str):
    cands = sorted(glob.glob(pattern))
    if not cands:
        raise FileNotFoundError(
            f"找不到特征文件：{pattern}\n"
            f"请确认你保存的 xlsx 文件名，或修改 FEATURE_FILE_GLOB。"
        )
    return cands[-1]

def find_smiles_col(df: pd.DataFrame):
    cand = [c for c in df.columns if isinstance(c, str) and "smiles" in c.lower()]
    if not cand:
        return None
    for p in ["Canonical SMILES", "canonical_smiles", "SMILES", "smiles"]:
        for c in cand:
            if c.lower() == p.lower():
                return c
    return cand[0]

def is_binary_01_series(s: pd.Series) -> bool:
    if s.dtype == bool:
        return True
    if not np.issubdtype(s.dtype, np.number):
        return False
    vals = pd.unique(s.dropna())
    if len(vals) == 0:
        return False
    return set(vals).issubset({0, 1})

def infer_feature_cols_morgan(df: pd.DataFrame):
    cols = []
    for prefix in ["Rule__", "FG__", "morgan_", "fp_", "ecfp_", "mfp_", "bit_", "KG_emb_"]:
        cols.extend([c for c in df.columns if isinstance(c, str) and c.startswith(prefix)])
    if cols:
        return list(cols)

    num_cols = []
    for c in df.columns:
        if isinstance(c, (int, np.integer)):
            num_cols.append(c)
        elif isinstance(c, str) and c.isdigit():
            num_cols.append(c)

    if num_cols:
        return sorted(num_cols, key=lambda x: int(x))

    raise ValueError("未找到特征列（Rule__/FG__/morgan_/bit_/KG_emb_ 或 0..2047 纯数字列）。")

def infer_label_cols(df: pd.DataFrame, smiles_col: str, feature_cols: list):
    exclude = set(feature_cols)
    if smiles_col is not None:
        exclude.add(smiles_col)

    label_cols = []
    for c in df.columns:
        if c in exclude:
            continue
        if is_binary_01_series(df[c]):
            label_cols.append(c)

    if not label_cols:
        raise ValueError("未识别到标签列（0/1）。请确认文件中仍包含标签列。")
    return label_cols

def load_Xy_from_transformed_morgan(xlsx_path: str):
    df = pd.read_excel(xlsx_path)

    smiles_col = find_smiles_col(df)
    feature_cols = infer_feature_cols_morgan(df)
    label_cols = infer_label_cols(df, smiles_col, feature_cols)

    X = df[feature_cols].fillna(0).astype(np.float32).values
    y = df[label_cols].fillna(0).astype(int).values

    if X.shape[0] != y.shape[0]:
        raise ValueError("X 和 y 行数不一致，请检查文件。")

    return X, y, feature_cols, label_cols, df

# =========================
# 3) 指标（macro）
# =========================
def multilabel_macro_metrics(y_true, y_prob, thresh=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= thresh).astype(int)

    L = y_true.shape[1]
    accs, precs, recs, specs = [], [], [], []
    aurocs, auprcs = [], []

    for k in range(L):
        yt = y_true[:, k]
        yp = y_pred[:, k]
        ys = y_prob[:, k]

        tp = int(np.sum((yt == 1) & (yp == 1)))
        tn = int(np.sum((yt == 0) & (yp == 0)))
        fp = int(np.sum((yt == 0) & (yp == 1)))
        fn = int(np.sum((yt == 1) & (yp == 0)))
        n = len(yt)

        accs.append((tp + tn) / n if n else np.nan)
        precs.append(tp / (tp + fp) if (tp + fp) else np.nan)
        recs.append(tp / (tp + fn) if (tp + fn) else np.nan)
        specs.append(tn / (tn + fp) if (tn + fp) else np.nan)

        if len(np.unique(yt)) == 2:
            aurocs.append(roc_auc_score(yt, ys))
            auprcs.append(average_precision_score(yt, ys))
        else:
            aurocs.append(np.nan)
            auprcs.append(np.nan)

    def nanmean(x):
        return float(np.nanmean(np.asarray(x, dtype=float)))

    return {
        "Accuracy_macro": nanmean(accs),
        "Precision_macro": nanmean(precs),
        "Recall_macro": nanmean(recs),
        "Specificity_macro": nanmean(specs),
        "AUROC_macro": nanmean(aurocs),
        "AUPRC_macro": nanmean(auprcs),
        "AUROC_valid_labels": int(np.sum(~np.isnan(aurocs))),
        "AUPRC_valid_labels": int(np.sum(~np.isnan(auprcs))),
        "n_labels": int(L),
    }

def extract_positive_proba(p, n_labels: int, classes_list=None):
    """
    MultiOutputClassifier.predict_proba -> list length=L
    兼容：某些标签在某 fold 训练集只有单类 => (n,1)
    """
    if not isinstance(p, list):
        raise ValueError("MultiOutputClassifier.predict_proba 预期返回 list，但未得到 list。")

    out = np.zeros((p[0].shape[0], n_labels), dtype=np.float32)
    for k in range(n_labels):
        pk = p[k]
        if pk.ndim != 2:
            raise ValueError(f"predict_proba[{k}] 形状异常: {pk.shape}")

        if pk.shape[1] == 2:
            if classes_list is not None and len(classes_list) == n_labels:
                cls = list(classes_list[k])
                out[:, k] = pk[:, cls.index(1)].astype(np.float32) if 1 in cls else 0.0
            else:
                out[:, k] = pk[:, 1].astype(np.float32)

        elif pk.shape[1] == 1:
            if classes_list is not None and len(classes_list) == n_labels:
                only_cls = int(list(classes_list[k])[0])
                out[:, k] = 1.0 if only_cls == 1 else 0.0
            else:
                out[:, k] = 0.0
        else:
            raise ValueError(f"predict_proba[{k}] 类别数异常: {pk.shape[1]}")
    return out

# =========================
# 4) 近似分层 5 折：用 label cardinality 分层
# =========================
def make_stratify_target(y: np.ndarray, n_bins: int = 10):
    card = y.sum(axis=1).astype(int)
    uniq = np.unique(card)
    if len(uniq) <= 15:
        return card
    r = pd.Series(card).rank(method="average").values
    bins = pd.qcut(r, q=min(n_bins, len(np.unique(r))), labels=False, duplicates="drop")
    return np.asarray(bins, dtype=int)

def build_folds(X, y, n_splits=5, seed=42):
    strat_y = make_stratify_target(y)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return list(skf.split(X, strat_y))

# =========================
# 5) 五折CV评估（用固定 best params）
# =========================
def cv_eval_bestparams(X, y, folds, params, thresh=0.5):
    n_labels = y.shape[1]
    fold_rows = []

    for fold_id, (tr_idx, va_idx) in enumerate(folds, start=1):
        X_tr, X_va = X[tr_idx], X[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        base_est = lgb.LGBMClassifier(**params)
        clf = MultiOutputClassifier(base_est, n_jobs=1)
        clf.fit(X_tr, y_tr)

        p = clf.predict_proba(X_va)
        y_prob = extract_positive_proba(p, n_labels=n_labels, classes_list=getattr(clf, "classes_", None))

        m = multilabel_macro_metrics(y_va, y_prob, thresh=thresh)
        m["fold"] = fold_id
        fold_rows.append(m)

        print(
            f"[FOLD {fold_id}] "
            f"AUPRC={m['AUPRC_macro']:.6f} | AUROC={m['AUROC_macro']:.6f} | "
            f"Acc={m['Accuracy_macro']:.6f} | P={m['Precision_macro']:.6f} | "
            f"R={m['Recall_macro']:.6f} | Spec={m['Specificity_macro']:.6f}"
        )

    return pd.DataFrame(fold_rows).sort_values("fold")

# =========================
# 6) mean+_CI95（小样本用t；无scipy则退回1.96）
# =========================
def mean_ci95(arr: np.ndarray):
    arr = np.asarray(arr, dtype=float)
    arr = arr[~np.isnan(arr)]
    n = len(arr)
    if n == 0:
        return np.nan, np.nan, np.nan, 0

    mean = float(np.mean(arr))
    std = float(np.std(arr, ddof=1)) if n >= 2 else 0.0
    se = std / np.sqrt(n) if n > 0 else np.nan

    try:
        import scipy.stats as st
        tcrit = float(st.t.ppf(0.975, df=n-1)) if n >= 2 else 1.96
    except Exception:
        tcrit = 1.96

    half = tcrit * se if n >= 2 else 0.0
    return mean, mean - half, mean + half, n

def format_mean_plus_ci95(mean, lo, hi):
    if np.isnan(mean):
        return "nan"
    half = (hi - lo) / 2.0
    return f"{mean:.6f}+_{half:.6f}"

def summarize_mean_ci95(folds_df: pd.DataFrame):
    metric_cols = [
        "Accuracy_macro",
        "Precision_macro",
        "Recall_macro",
        "Specificity_macro",
        "AUROC_macro",
        "AUPRC_macro",
    ]
    rows = []
    for col in metric_cols:
        mean, lo, hi, n = mean_ci95(folds_df[col].values)
        rows.append({
            "metric": col,
            "mean": mean,
            "ci95_low": lo,
            "ci95_high": hi,
            "n_folds": n,
            "mean+_CI95": format_mean_plus_ci95(mean, lo, hi),
        })
    return pd.DataFrame(rows)

# =========================
# 7) 主流程
# =========================
def main():
    feature_file = find_feature_file(FEATURE_FILE_GLOB)
    print("[INFO] Using transformed feature file:", feature_file)

    X, y, feat_cols, label_cols, _df_all = load_Xy_from_transformed_morgan(feature_file)
    print(f"[INFO] X shape={X.shape} (features={len(feat_cols)}) | y shape={y.shape} (labels={len(label_cols)})")

    # ✅ 合并固定参数 + 手写 best params
    best_params = dict(BASE_LGB_PARAMS)
    best_params.update(BEST_PARAMS)

    print("\n[INFO] Using fixed best params (directly specified):")
    print(json.dumps(best_params, indent=2, ensure_ascii=False, default=str))

    folds = build_folds(X, y, n_splits=N_SPLITS, seed=RANDOM_SEED)
    print(f"\n[INFO] Prepared fixed {N_SPLITS}-fold splits.")

    folds_df = cv_eval_bestparams(X, y, folds, best_params, thresh=THRESH)
    folds_df.to_csv(OUT_FOLDS_CSV, index=False, encoding="utf-8-sig")
    print(f"\n[SAVED] {OUT_FOLDS_CSV}")

    sum_df = summarize_mean_ci95(folds_df)
    sum_df.to_csv(OUT_MEANCI_CSV, index=False, encoding="utf-8-sig")
    print(f"[SAVED] {OUT_MEANCI_CSV}")

    print("\n========== 5-FOLD MEAN+_CI95 ==========")
    for _, r in sum_df.iterrows():
        print(f"{r['metric']}: {r['mean+_CI95']}")

    # 全量训练并保存
    base_est = lgb.LGBMClassifier(**best_params)
    clf_full = MultiOutputClassifier(base_est, n_jobs=1)
    clf_full.fit(X, y)

    joblib.dump(
        {
            "model": clf_full,
            "params": best_params,
            "feature_cols": feat_cols,
            "label_cols": label_cols,
            "threshold": THRESH,
            "random_seed": RANDOM_SEED,
        },
        OUT_MODEL_FILE
    )
    print(f"\n[SAVED] full-fit model -> {OUT_MODEL_FILE}")

if __name__ == "__main__":
    main()


[INFO] Using transformed feature file: ./Malodors_transformed_MORGAN_features.xlsx
[INFO] X shape=(4952, 2048) (features=2048) | y shape=(4952, 138) (labels=138)

[INFO] Using fixed best params (directly specified):
{
  "objective": "binary",
  "boosting_type": "gbdt",
  "n_jobs": -1,
  "random_state": 42,
  "verbosity": -1,
  "n_estimators": 507,
  "learning_rate": 0.04407984038169244,
  "num_leaves": 38,
  "max_depth": 19,
  "min_child_samples": 24,
  "subsample": 0.8650089137415928,
  "colsample_bytree": 0.7246844304357644,
  "reg_lambda": 0.2778331300797557,
  "reg_alpha": 0.0008325158565947976,
  "min_split_gain": 0.9242722776276352
}

[INFO] Prepared fixed 5-fold splits.
[FOLD 1] AUPRC=0.297176 | AUROC=0.846327 | Acc=0.972733 | P=0.592345 | R=0.108353 | Spec=0.995441
[FOLD 2] AUPRC=0.314382 | AUROC=0.854661 | Acc=0.972894 | P=0.665749 | R=0.112739 | Spec=0.995604
[FOLD 3] AUPRC=0.291315 | AUROC=0.852091 | Acc=0.972727 | P=0.676303 | R=0.107790 | Spec=0.995581
[FOLD 4] AUPRC=0.307